<a href="https://colab.research.google.com/github/lsmyk/libraryJuiceAcademyAssignment/blob/main/Matching_Alma_to_OCLC_records_LAST_update.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
import csv
import unicodedata
from collections import defaultdict

In [2]:
import re
import csv
import unicodedata
from collections import defaultdict

# ---------- Helpers ----------

def normalize_oclc(field):
    if not field:
        return ""

    # Extract ONLY current OCLC number from $a
    match = re.search(r"\$a\(OCoLC\)(\d+)", field)

    if match:
        return match.group(1)

    return ""


def normalize_text(text): #Normalizes titles and author's names
    if not text:
        return ""

    text = unicodedata.normalize('NFKD', text) #Removes accents(Gaillárd → Gaillard)
    text = ''.join(c for c in text if not unicodedata.combining(c))

    text = re.sub(r"[^\w\s]", " ", text) #Removes punctuation
    text = re.sub(r"\s+", " ", text) #Collapses multiple spaces

    return text.strip().lower() #Convert to lowercase

def normalize_edition(ed):
    if not ed:
        return ""
    ed = ed.lower()#Convert to lowercase

    # ✅ remove backslashes early
    ed = ed.replace("\\", "")

    # ✅ remove subfield delimiters and letters ($a, $b, $c)
    ed = re.sub(r"\$[a-z]", " ", ed)

    # ✅ remove brackets explicitly
    ed = re.sub(r"[\[\]]", "", ed)

    # ✅ normalize ordinal words/numbers
    replacements = {
      "first": "1", "1st": "1",
      "second": "2", "2d": "2", "2nd": "2",
      "third": "3", "3d": "3", "3rd": "3",
      "fourth": "4", "4th": "4"
    }
    for k, v in replacements.items():
        ed = re.sub(rf"\b{k}\b", v, ed)

    # remove leading 'a' (from $a)
    ##ed = re.sub(r"^a", "", ed)

    # ✅ normalize key terms
    ed = ed.replace("edition", "ed")
    ed = ed.replace("revised", "rev")
    ed = ed.replace("rev.", "rev")
    ed = ed.replace("enlarged", "enl")

    # ✅ remove long descriptive phrases
    ed = re.sub(r"with.*", "", ed)

    # ✅ remove connectors
    ed = re.sub(r"\b(and|with)\b", " ", ed)

    # ✅ remove punctuation
    ed = re.sub(r"[^\w\s]", "", ed)

    # ✅ normalize spaces
    ed = re.sub(r"\s+", " ", ed)

    # clean if previous ediiting created extra spaces
    ed = re.sub(r"[^\w\s]", "", ed)
    ed = re.sub(r"\s+", " ", ed)

    return ed.strip()

def normalize_date(raw):
    if not raw:
        return ""

    raw = raw.lower()

    # remove brackets and symbols
    raw = re.sub(r"[\[\]\(©c\?\.]", "", raw)

    # extract all 4-digit years
    years = re.findall(r"\d{4}", raw)

    if not years:
        return ""

    return years[0]

  # take first year (publication priority)

# ✅ DEFINE word_overlap AT TOP LEVEL (no indentation!)
def word_overlap(a, b):
    if not a or not b:
        return 0

    stopwords = {"a", "the", "of", "and", "in", "de", "der", "et"}

    set_a = {w for w in a.split() if w not in stopwords}
    set_b = {w for w in b.split() if w not in stopwords}

    if not set_a or not set_b:
        return 0

    return len(set_a & set_b) / min(len(set_a), len(set_b))



# ---------- MRK Parser ----------

def parse_mrk(file_path):
    records = []
    current = defaultdict(list)

    with open(file_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line.startswith("=LDR"):
                if current:
                    records.append(current)
                    current = defaultdict(list)
            elif line.startswith("="):
                tag = line[1:4]
                value = line[6:]
                current[tag].append(value)

        if current:
            records.append(current)

    return records

# ---------- Extract Fields ----------

def extract_245_title(field):
    if not field:
        return ""

    field = field.strip()

    # Remove indicators (e.g. "14", "10")
    field = re.sub(r"^\d\d", "", field).strip()

    # Extract subfield $a (always present in your data)
    match = re.search(r"\$a\s*([^$]+)", field)
    if match:
        return match.group(1).strip()

    return ""  # fallback (should not happen in your case)

def extract_record_data(rec, is_alma=True):
    data = {}

    # MMSID
    data["mmsid"] = rec.get("001", [""])[0]

    # OCLC
    current_oclc = ""
    old_oclcs = []

    for field_035 in rec.get("035", []):
        # Attempt to find the current OCLC number first
        # We assume the first 035 field that matches the pattern for a current OCLC is the primary one.
        if not current_oclc: # Only try to set if not already found
            potential_current = normalize_oclc(field_035)
            if potential_current:
                current_oclc = potential_current

        # Extract all (OCoLC) numbers from fields containing '$z' for superseded OCLC numbers
        nums = re.findall(r"\(OCoLC\)(\d+)", field_035)
        if "$z" in field_035:
            for num in nums:
                old_oclcs.append(num)

    data["oclc"] = current_oclc # Assign the found current OCLC, or "" if none
    data["old_oclc"] = old_oclcs

    # Title
    raw_title = rec["245"][0] if "245" in rec else ""
    title = extract_245_title(raw_title)
    data["title"] = normalize_text(title)

    # Author
    author = ""
    for tag in ["100", "110", "111", "130"]:
        if tag in rec:
            author = rec[tag][0]
            break
    # remove indicators
    author = re.sub(r"^\d+\s*", "", author)

    # remove $a
    author = re.sub(r"\$a", "", author)

    # remove unwanted subfields
    author = re.sub(r"\$e[^$]*", "", author)
    author = re.sub(r"\$1[^$]*", "", author)
    author = re.sub(r"\$0[^$]*", "", author)

    # normalize
    data["author"] = normalize_text(author)

    # Edition
    edition = rec.get("250", [""])[0]
    data["edition"] = normalize_edition(edition)

    # Publication date (raw)
    date_fields = rec.get("260", []) + rec.get("264", [])

    raw_date = ""

    for field in date_fields:
        # ✅ first try $c (preferred, OCLC + standard MARC)
        match = re.search(r"\$c([^$]+)", field)
        if match:
            raw_date = match.group(1)
            break

    # ✅ fallback to $a (Alma-specific behavior)
    if not raw_date:
        for field in date_fields:
            match = re.search(r"\$a([^$]+)", field)
            if match:
                raw_date = match.group(1)
                break

    data["date_raw"] = raw_date
    data["date"] = normalize_date(raw_date)


    return data


# ---------- Matching Logic ----------

def compare_records(alma, oclc):

    # ✅ 1. Edition check (hard rule)
    if alma["edition"] and oclc["edition"]:
        if alma["edition"] != oclc["edition"]:
            return "NO", "edition mismatch"

    # ✅ 2. Date comparison (informational only)
    date_note = ""

    if alma["date"] and oclc["date"]:
        if alma["date"] != oclc["date"]:
            date_note = f"date mismatch ({alma['date']} vs {oclc['date']})"

    elif alma["date"] or oclc["date"]:
        date_note = "date incomplete or uncertain"

    # ✅ 3. Compute scores
    title_score = word_overlap(alma["title"], oclc["title"])
    author_score = word_overlap(alma["author"], oclc["author"])

    # ✅ 4. Decision
    if title_score >= 0.6:
        if author_score >= 0.5:
            reason = f"title+author ({title_score:.2f}/{author_score:.2f})"
        else:
            reason = f"title only ({title_score:.2f})"

        # ✅ append date info if present
        if date_note:
            reason += " - " + date_note

        return "YES", reason

    # ✅ fallback
    return "NO", f"title mismatch ({title_score:.2f})"


# ---------- Main ----------

alma_records = parse_mrk("/content/ALMA.mrk")
oclc_records = parse_mrk("/content/OCLC.mrk")



# Build lookup by OCLC#
oclc_index = {}

for rec in oclc_records:

    if "245" in rec:
        raw = rec["245"][0]

        extracted = extract_245_title(raw)
        normalized = normalize_text(extracted)

    # existing logic
    if "245" not in rec:
        print("No 245 in record:", rec.get("001", "unknown"))

    data = extract_record_data(rec, is_alma=False)

    if data["oclc"]:
        oclc_index[data["oclc"]] = data

        for old in data.get("old_oclc", []):
            oclc_index[old] = data


oclc_keys = set(oclc_index.keys())
alma_keys = set()

for rec in alma_records:
    data = extract_record_data(rec, is_alma=True)
    if data["oclc"]:
        alma_keys.add(data["oclc"])

intersection = oclc_keys & alma_keys


# Compare
results = []

for rec in alma_records:
    alma_data = extract_record_data(rec, is_alma=True)

    oclc_data = oclc_index.get(alma_data["oclc"])
    confidence = 0

    # ✅ STEP 1: exact OCLC match
    if oclc_data:
        match, reason = compare_records(alma_data, oclc_data)
        match_type = "OCLC"
        confidence = 1.0

    # ✅ STEP 2: Title match
    else:
        best_matches = []
        best_score = 0

        for candidate in oclc_index.values():
            score = word_overlap(alma_data["title"], candidate["title"])

            if score > best_score:
                best_score = score
                best_matches = [candidate]

            elif score == best_score:
                best_matches.append(candidate)

        # ✅ Choose best among top equals
        best_match = None

        for candidate in best_matches:
            if word_overlap(alma_data["author"], candidate["author"]) > 0.5:
                best_match = candidate
                break

        if not best_match and best_matches:
            best_match = best_matches[0]


        # ✅ final decision
        if best_match and best_score >= 0.6:
            oclc_data = best_match
            match, reason = compare_records(alma_data, oclc_data)
            reason = f"title match ({best_score:.2f}) - " + reason
            match_type = "TITLE"
            confidence = best_score
        else:
            match = "NO"
            reason = "no OCLC or title match found"
            match_type = "NONE"
            confidence = 0

    # OCLC comparison
    if not oclc_data:
        oclc_status = "NO OCLC RECORD"

    elif alma_data["oclc"] == oclc_data["oclc"]:
        oclc_status = "CURRENT"

    elif alma_data["oclc"] in oclc_data.get("old_oclc", []):
        oclc_status = "FOUND IN 035$z"

    else:
        oclc_status = "NOT FOUND"


    results.append([
        alma_data["mmsid"],
        alma_data["oclc"],                       # Alma OCLC
        oclc_data["oclc"] if oclc_data else "", # Retrieved OCLC
        oclc_status,
        match,
        confidence,
        reason,
        alma_data["title"],
        oclc_data["title"] if oclc_data else "",
        alma_data["author"],
        oclc_data["author"] if oclc_data else "",
        alma_data["edition"],
        oclc_data["edition"] if oclc_data else "",
        alma_data["date"],
        oclc_data["date"] if oclc_data else ""

    ])


# Export
with open("comparison_results.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
    "MMSID", "Alma 035$a","OCLC 035$a","OCLC Relationship", "Match", "Confidence", "Reason",
    "Alma Title", "OCLC Title",
    "Alma Author", "OCLC Author",
    "Alma 250", "OCLC 250",
    "Alma Date", "OCLC Date"
    ])

    writer.writerows(results)

print("Done! Output saved to comparison_results.csv")

Done! Output saved to comparison_results.csv
